In [0]:
# Konfiguracja — projekt i dostawca
dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca
blueprint_path = f"/Volumes/{catalog}/{schema}/blueprint/"

print(f"Notebook 00 — Ingest Blueprint")
print(f"Dostawca:  {dostawca}")
print(f"Ścieżka:   {blueprint_path}")

# Kod ustawia dwa widgety: 'projekt' i 'dostawca', pobiera ich wartości, buduje ścieżkę blueprint_path i wyświetla informacje o dostawcy oraz ścieżce.

In [0]:
%pip install pdfplumber openai
dbutils.library.restartPython()


In [0]:
# Ustawia widgety, pobiera wartości, buduje ścieżkę blueprint_path i pobiera klucz OpenAI z AzureADLS.
dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca
blueprint_path = f"/Volumes/{catalog}/{schema}/blueprint/"

openai_key = dbutils.secrets.get("openai", "tokenGPT")

print(f"Dostawca:  {dostawca}")
print(f"Ścieżka:   {blueprint_path}")
print(f"OpenAI key: {'*' * 10}{openai_key[-4:]}")

# Test czy klucz został pobrany
if openai_key and len(openai_key) > 0:
    print("Klucz OpenAI został poprawnie pobrany.")
else:
    print("Błąd: Klucz OpenAI nie został pobrany.")

In [0]:
# === SZUKANIE PDF KONTRAKTU ===
# Listujemy pliki w folderze blueprint i szukamy PDF-ów
pliki = dbutils.fs.ls(blueprint_path)
pliki_pdf = [f for f in pliki if f.name.lower().endswith(".pdf")]

if len(pliki_pdf) == 0:
    raise Exception(f"Brak pliku PDF w {blueprint_path}")

# Wybieramy najnowszy PDF
najnowszy = sorted(pliki_pdf, key=lambda f: f.modificationTime, reverse=True)[0]
sciezka_pdf = najnowszy.path.replace("dbfs:", "")

print(f"Znaleziono {len(pliki_pdf)} plik(ów) PDF")
print(f"Wybrany:   {najnowszy.name}")


In [0]:
# === KONFIGURACJA ===
# Widgety pozwalają zmienić projekt/dostawcę bez edycji kodu
dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

# Ścieżka do folderu gdzie leży PDF kontraktu
catalog = projekt
schema = dostawca
blueprint_path = f"/Volumes/{catalog}/{schema}/blueprint/"

# Klucz OpenAI z Databricks Secrets — nigdy nie pojawia się w kodzie wprost
openai_key = dbutils.secrets.get("openai", "tokenGPT")

print(f"Dostawca:  {dostawca}")
print(f"Ścieżka:   {blueprint_path}")
print(f"Klucz OK:  {'*' * 10}{openai_key[-4:]}")





In [0]:
import pdfplumber

# === WYCIĄGANIE TEKSTU I TABEL Z PDF ===
# pdfplumber czyta tekst + tabele osobno — tabele formatujemy jako markdown
tekst_kontraktu = ""

with pdfplumber.open(sciezka_pdf) as pdf:
    for i, strona in enumerate(pdf.pages):
        tekst_kontraktu += f"\n--- Strona {i+1} ---\n"
        
        # Wyciągamy tekst
        tekst = strona.extract_text()
        if tekst:
            tekst_kontraktu += tekst + "\n"
        
        # Wyciągamy tabele jako markdown
        tabele = strona.extract_tables()
        for t, tabela in enumerate(tabele):
            tekst_kontraktu += f"\n[TABELA {t+1}]\n"
            for wiersz in tabela:
                komorki = [str(k).replace("\n", " ").strip() if k else "" for k in wiersz]
                tekst_kontraktu += " | ".join(komorki) + "\n"

print(f"Łączna liczba znaków: {len(tekst_kontraktu)}")
print(tekst_kontraktu[:1000])


In [0]:
import pdfplumber

# === WYCIĄGANIE TEKSTU Z PDF ===
# pdfplumber czyta każdą stronę i skleja tekst w jeden string
tekst_kontraktu = ""
with pdfplumber.open(sciezka_pdf) as pdf:
    for i, strona in enumerate(pdf.pages):
        tekst = strona.extract_text()
        if tekst:
            tekst_kontraktu += f"\n--- Strona {i+1} ---\n{tekst}"

print(f"Wyciągnięto tekst z {len(pdf.pages)} stron")
print(f"Łączna liczba znaków: {len(tekst_kontraktu)}")
print("\nPodgląd (pierwsze 500 znaków):")
print(tekst_kontraktu[:500])


In [0]:
from openai import OpenAI
import json

client = OpenAI(api_key=openai_key)

prompt = f"""Jesteś ekspertem ds. analizy kontraktów.
Przeanalizuj poniższy kontrakt i zwróć dane dostawcy w formacie JSON.

Zwróć TYLKO JSON (bez żadnego tekstu przed ani po):
{{
  "dostawca_id": "nazwa_firmy_małymi_literami_bez_spacji",
  "nazwa_pelna": "...",
  "nip": "...",
  "adres": "...",
  "miasto": "...",
  "kod_pocztowy": "...",
  "kontrakt_od": "YYYY-MM-DD",
  "kontrakt_do": "YYYY-MM-DD"
}}

Kontrakt:
{tekst_kontraktu}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

# Usuwamy znaczniki markdown jeśli GPT je dodał
tekst = response.choices[0].message.content.strip()
if tekst.startswith("```"):
    tekst = tekst.split("```")[1]
    if tekst.startswith("json"):
        tekst = tekst[4:]

dane_dostawcy = json.loads(tekst.strip())
print(json.dumps(dane_dostawcy, indent=2, ensure_ascii=False))


In [0]:
from datetime import date

# === ZAPIS DANYCH DOSTAWCY DO TABELI ===
# Wstawiamy lub nadpisujemy wiersz dla tego dostawcy (MERGE po dostawca_id)
spark.sql(f"""
MERGE INTO inspektor_budzet.ops.dostawcy AS cel
USING (
  SELECT
    '{dane_dostawcy["dostawca_id"]}'  AS dostawca_id,
    '{dane_dostawcy["nazwa_pelna"]}'  AS nazwa_pelna,
    CAST('{dane_dostawcy["nip"].replace("-", "")}' AS BIGINT) AS nip,
    true                               AS aktywny,
    DATE('{dane_dostawcy["kontrakt_od"]}') AS kontrakt_od,
    DATE('{dane_dostawcy["kontrakt_do"]}') AS kontrakt_do,
    '{dane_dostawcy["adres"]}'         AS adres,
    '{dane_dostawcy["miasto"]}'        AS miasto,
    '{dane_dostawcy["kod_pocztowy"]}' AS kod_pocztowy,
    NULL AS email_kontakt,
    NULL AS telefon,
    NULL AS notatki,
    'inspektor_budzet'                 AS catalog_name,
    '{dostawca}'                       AS schema_name
) AS nowe
ON cel.dostawca_id = nowe.dostawca_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"Zapisano dostawcę: {dane_dostawcy['nazwa_pelna']} ✅")


In [0]:
# === ANALIZA KONTRAKTU — POZYCJE I REGUŁY ===
prompt = f"""Jesteś ekspertem ds. analizy kontraktów dla jednostek samorządowych.
Przeanalizuj poniższy kontrakt i zwróć pozycje rozliczeniowe oraz reguły kalkulacji w formacie JSON.

Typy pozycji:
- fixed: stała opłata niezależna od ilości (np. ryczałt miesięczny)
- linear: cena × ilość, jedna stawka dla całego zakresu
- tiered: cena zależy od ilości — różne stawki dla różnych przedziałów
- conditional: naliczana tylko gdy spełniony dodatkowy warunek zewnętrzny

Typy reguł (rule_type):
- FIXED: jedna stała kwota, cena = wartość ryczałtu, prog_od i prog_do = null
- LINEAR: cena × ilość, prog_od i prog_do = null
- TIERED: jeden wiersz na przedział cenowy, prog_od i prog_do określają przedział
- GATE: warunek który musi być spełniony — cena = null, warunek = opis warunku

ZASADY:
1. Każda pozycja MUSI mieć co najmniej jedną regułę.
2. Dla tiered: JEDNA pozycja reprezentuje całą usługę (np. "Wykop rowu"), 
   a każdy przedział cenowy to osobna reguła TIERED tej samej pozycji.
   NIGDY nie twórz osobnych pozycji dla różnych progów tej samej usługi.
   Pierwszy próg zawsze prog_od = 0. Kolejny próg zaczyna się od prog_do poprzedniego.
3. Dla conditional: DWE reguły dla tej samej pozycji:
   - GATE (kolejnosc=1): opisuje warunek który musi być spełniony i jak jest weryfikowany 
     (np. wymagany dokument, podpis, protokół). cena = null.
   - LINEAR (kolejnosc=2): kalkulacja gdy warunek spełniony. cena = kwota z kontraktu.
4. Dla fixed i linear: jedna reguła z rzeczywistą ceną z kontraktu.
5. NIGDY nie wstawiaj cena: 0.0 — zawsze wyciągaj kwotę z tekstu. Jeśli nie ma ceny, wstaw null.
6. Wyciągnij WSZYSTKIE pozycje rozliczeniowe z kontraktu — nie pomijaj żadnej. 
   Sprawdź każdą pozycję w tabeli rozliczeniowej i upewnij się że jest w JSON.



Zwróć TYLKO JSON:
{{
  "pozycje": [
    {{
      "position_id": "WIELKIE_LITERY_BEZ_SPACJI",
      "nazwa": "...",
      "jednostka": "...",
      "typ": "fixed|linear|tiered|conditional"
    }}
  ],
  "reguly": [
    {{
      "rule_id": "R001",
      "position_id": "...",
      "kolejnosc": 1,
      "rule_type": "FIXED|LINEAR|TIERED|GATE",
      "prog_od": null,
      "prog_do": null,
      "cena": 0.0,
      "warunek": null,
      "opis": "krótki opis tej reguły zrozumiały dla użytkownika"
    }}
  ]
}}

Kontrakt:
{tekst_kontraktu}"""

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}]
)

tekst = response.choices[0].message.content.strip()
if tekst.startswith("```"):
    tekst = tekst.split("```")[1]
    if tekst.startswith("json"):
        tekst = tekst[4:]

dane_kontraktu = json.loads(tekst.strip())
print(json.dumps(dane_kontraktu, indent=2, ensure_ascii=False))




In [0]:
# === ZAPIS POZYCJI DO TABELI ===
spark.sql(f"DELETE FROM inspektor_budzet.{schema}.blueprint_pozycje")

for p in dane_kontraktu["pozycje"]:
    spark.sql(f"""
    INSERT INTO inspektor_budzet.{schema}.blueprint_pozycje VALUES (
        '{p["position_id"]}',
        '{p["nazwa"]}',
        '{p["jednostka"]}',
        '{p["typ"]}'
    )
    """)

print(f"Zapisano {len(dane_kontraktu['pozycje'])} pozycji ✅")

# === ZAPIS REGUŁ DO TABELI ===
spark.sql(f"DELETE FROM inspektor_budzet.{schema}.blueprint_reguly")

for r in dane_kontraktu["reguly"]:
    cena = r["cena"] if r["cena"] is not None else "NULL"
    prog_od = r["prog_od"] if r["prog_od"] is not None else "NULL"
    prog_do = r["prog_do"] if r["prog_do"] is not None else "NULL"
    warunek = r["warunek"].replace("'", "''") if r["warunek"] else ""
    opis = r["opis"].replace("'", "''") if r["opis"] else ""

    spark.sql(f"""
    INSERT INTO inspektor_budzet.{schema}.blueprint_reguly VALUES (
        '{r["rule_id"]}',
        '{r["position_id"]}',
        {r["kolejnosc"]},
        {prog_od},
        {prog_do},
        {cena},
        '{warunek}',
        '{opis}',
        '{r["rule_type"]}'
    )
    """)

print(f"Zapisano {len(dane_kontraktu['reguly'])} reguł ✅")
display(spark.table(f"inspektor_budzet.{schema}.blueprint_reguly"))
